# CH14 — Grafos (parte 1): conceptos, ADT y la implementación de Goodrich

Material del curso basado en Goodrich, Tamassia & Goldwasser — Capítulo 14 (secciones 14.1 y 14.2).

**Objetivo:** conocer el vocabulario de los grafos, el ADT `Graph` que usa Goodrich, las cuatro formas clásicas de guardar un grafo en memoria y la implementación con **mapa de adyacencia** que usaremos en el resto del capítulo (recorridos, caminos más cortos, árboles de expansión mínima).

**Contenido**

0. ¿Para qué sirven los grafos? Aplicaciones
1. ¿Qué es un grafo? Terminología
2. Propiedades básicas
3. El ADT `Graph`
4. Estructuras de datos para grafos
5. La implementación de Goodrich: mapa de adyacencia
6. Ejemplos de uso
7. Resumen

<p align='center'>
    <img src = 'https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b9/WorldWideWebAroundWikipedia.png/960px-WorldWideWebAroundWikipedia.png?utm_source=es.wikipedia.org&utm_campaign=parser&utm_content=thumbnail'>
</p>

## 0. ¿Para qué sirven los grafos? Aplicaciones

Un grafo modela **cosas** (vértices) y **relaciones entre ellas** (aristas). Esa idea tan simple aparece en buscadores, mapas, redes sociales, horarios, compiladores, redes eléctricas... Antes de las definiciones formales, veamos algunas aplicaciones conocidas. En cada una, la pregunta clave es: **¿qué son los vértices y qué son las aristas?**

![Aplicaciones de los grafos](../assets/ch14_aplicaciones.png)

### PageRank: el algoritmo original de Google

- **Vértices:** páginas web. **Aristas dirigidas:** un enlace de la página $u$ a la página $v$.
- Idea de Brin y Page (1998): una página es importante si la enlazan páginas importantes. Cada página **reparte** su importancia en partes iguales entre las páginas a las que enlaza (su $\text{outdeg}$), y recibe importancia por sus enlaces entrantes (su $\text{indeg}$).
- Equivale a un "navegante aleatorio": con probabilidad $d = 0.85$ sigue un enlace al azar de la página actual, y con probabilidad $0.15$ salta a cualquier página. El PageRank de una página es la fracción del tiempo que el navegante pasa en ella.
- En la figura, `C` tiene el mayor $\text{indeg}$ (4) y el mayor PageRank. Pero `A`, con solo 2 enlaces entrantes, casi la iguala, porque uno de ellos viene de `C`: **no solo importa cuántos te enlazan, sino quién te enlaza**.

### Waze / Google Maps: caminos más cortos

- **Vértices:** esquinas o intersecciones. **Aristas:** tramos de calle, con un **peso** igual al tiempo estimado de recorrido. Las calles de un solo sentido son aristas dirigidas.
- "¿Cuál es la ruta más rápida de la casa a la universidad?" es el problema del **camino más corto**, que resuelve el algoritmo de **Dijkstra** (sección 14.6 de Goodrich). Las aplicaciones reales usan variantes más rápidas (A*, preprocesamiento del mapa), pero la base es la misma.
- Lo que hace especial a Waze es que **los pesos cambian**: los usuarios reportan trancones y accidentes, el peso de esa arista sube, y la ruta se recalcula. En la figura, la ruta "obvia" por `A → C` pasa por un trancón de 25 minutos; la mejor ruta es `Casa → B → D → U` (19 minutos).

### Coloración de grafos: horarios sin choques

- **Vértices:** materias. **Aristas:** dos materias tienen estudiantes en común, así que sus exámenes **no pueden** ser a la misma hora.
- **Colorear** el grafo es asignar un color a cada vértice de modo que los vértices adyacentes tengan colores distintos. Cada color es una franja horaria: el **mínimo número de colores** es el mínimo número de franjas.
- En la figura bastan 3 colores, y no se puede con menos: `Cál`, `Fís` y `Alg` forman un triángulo.
- Otros usos: colorear mapas (el **teorema de los cuatro colores**, 1976: todo mapa plano se puede colorear con 4 colores), asignar frecuencias a antenas de celular para que las vecinas no interfieran, y asignar registros del procesador en un compilador.
- Encontrar el mínimo número de colores es un problema **difícil** (NP-difícil). En la práctica se usan algoritmos voraces que dan buenas soluciones, aunque no siempre la óptima.

### Recorridos: los puentes de Königsberg

- El problema que dio origen a la teoría de grafos (Euler, 1736). La ciudad de Königsberg tenía 7 puentes que unían dos orillas (`N`, `S`), una isla (`I`) y otra zona (`E`). ¿Se puede dar un paseo que cruce **cada puente exactamente una vez**?
- **Vértices:** zonas de tierra. **Aristas:** puentes. Hay aristas paralelas: es un **multigrafo**.
- Euler demostró que un recorrido así (un **camino euleriano**) existe solo si el grafo es conexo y tiene **0 o 2 vértices de grado impar**. En Königsberg los cuatro vértices tienen grado impar (3, 5, 3, 3), así que es imposible. ¡La respuesta sale solo de mirar los **grados**!
- Aplicaciones actuales: rutas de camiones de basura, barrenderas o carteros que deben pasar por **todas las calles**.

### Recorrido en anchura (BFS): redes sociales

- **Vértices:** personas. **Aristas:** amistad (no dirigida, como en Facebook) o "seguir" (dirigida, como en Instagram o X).
- El recorrido en anchura (**BFS**, sección 14.3) explora el grafo **por niveles**: primero tus amigos (nivel 1), luego los amigos de tus amigos (nivel 2), etc. El nivel de cada persona es su **distancia** (en número de aristas) desde ti.
- Así funcionan las sugerencias de "personas que quizá conozcas" (nivel 2) y los "grados de separación". Según un estudio de Facebook (2016), dos usuarios cualesquiera están separados en promedio por unos **3.5** intermediarios.
- El otro recorrido, en profundidad (**DFS**), sirve para salir de un laberinto, detectar ciclos y averiguar qué se puede alcanzar desde un vértice.

### Orden topológico: prerrequisitos

- **Vértices:** materias. **Arista dirigida** $u \to v$: $u$ es prerrequisito de $v$.
- Si el grafo no tiene ciclos (es un **DAG**), existe un **orden topológico**: una forma de cursar las materias que respeta todos los prerrequisitos (sección 14.5). Si hubiera un ciclo, nadie podría graduarse.
- Mismo problema: el orden de compilación de los módulos de un programa, el orden de instalación de paquetes con dependencias (`pip`, `npm`) y la planeación de las tareas de un proyecto.

### Otras aplicaciones

| Aplicación | Vértices | Aristas | Problema / algoritmo | En Goodrich |
|---|---|---|---|---|
| Waze, Google Maps, GPS | intersecciones | calles con tiempo de recorrido | camino más corto (Dijkstra) | 14.6 |
| Red de fibra óptica, red eléctrica, acueducto | ciudades o edificios | posibles tendidos con su costo | conectar todo al mínimo costo: árbol de expansión mínima (Prim-Jarník, Kruskal) | 14.7 |
| Red social | personas | amistad / "seguir" | BFS, comunidades, recomendaciones | 14.3 |
| Laberintos, videojuegos | casillas | pasos permitidos | DFS / BFS, alcanzabilidad | 14.3, 14.4 |
| Plan de estudios, compilación, paquetes | materias, módulos | dependencias | orden topológico | 14.5 |
| Buscadores web | páginas | enlaces | PageRank | — |
| Horarios, frecuencias de antenas | materias, antenas | conflictos | coloración | — |
| Vuelos entre aeropuertos | aeropuertos | vuelos | alcanzabilidad, rutas con escalas | 14.1, 14.4 |
| Epidemiología | personas | contactos | propagación de un contagio (BFS por niveles) | 14.3 |
| Circuitos electrónicos | componentes | cables | diseño sin cruces (grafos planares) | — |

Lo que tienen en común: una vez que el problema se modela como grafo, **el mismo algoritmo** sirve para todos. Dijkstra no sabe si sus vértices son esquinas de Bogotá o routers de internet; solo ve vértices, aristas y pesos. Por eso vale la pena estudiar el ADT `Graph` y sus algoritmos de forma general.

#### Para probar: PageRank en pocas líneas

El PageRank de la figura se puede calcular repitiendo la regla del navegante aleatorio hasta que los valores se estabilicen. Las páginas se guardan como un diccionario `página -> lista de páginas a las que enlaza` (una lista de adyacencia, sección 4). Observa que cada página divide su importancia entre su $\text{outdeg}$: `len(salientes)`.

In [1]:
def pagerank(enlaces, d=0.85, iteraciones=100):
    """PageRank por iteración. enlaces: dict página -> lista de páginas a las que enlaza.

    Supone que toda página tiene al menos un enlace de salida (outdeg >= 1).
    """
    n = len(enlaces)
    pr = {p: 1 / n for p in enlaces}            # al inicio todas valen lo mismo
    for _ in range(iteraciones):
        nuevo = {p: (1 - d) / n for p in enlaces}   # el salto al azar le da a todas un poco
        for p, salientes in enlaces.items():
            for q in salientes:
                nuevo[q] += d * pr[p] / len(salientes)   # p reparte su importancia entre outdeg(p) páginas
        pr = nuevo
    return pr


ENLACES = {'A': ['B', 'C'], 'B': ['C'], 'C': ['A'], 'D': ['C', 'A'], 'E': ['C', 'D']}
pr = pagerank(ENLACES)
for p in sorted(pr, key=pr.get, reverse=True):
    print(f'{p}: {pr[p]:.1%}')
print(f'suma = {sum(pr.values()):.3f}')      # siempre 1: es una distribución de probabilidad
# C: 37.5%
# A: 36.7%
# B: 18.6%
# D: 4.3%
# E: 3.0%
# suma = 1.000

C: 37.5%
A: 36.7%
B: 18.6%
D: 4.3%
E: 3.0%
suma = 1.000


## 1. ¿Qué es un grafo? Terminología

Un **grafo** $G = (V, E)$ es un conjunto $V$ de **vértices** (*vertices*, también llamados nodos) y una colección $E$ de **aristas** (*edges*). Cada arista conecta un par de vértices.

- Una arista $(u, v)$ es **dirigida** si el par es **ordenado**: va de $u$ (**origen**) a $v$ (**destino**). Se dibuja con una flecha.
- Es **no dirigida** si el par **no** es ordenado: $(u, v)$ y $(v, u)$ son la misma arista. Se dibuja con una línea.
- Si todas las aristas son no dirigidas, el grafo es **no dirigido**; si todas son dirigidas, es un **grafo dirigido** o **digrafo**. Un grafo con ambos tipos es **mixto** (todo grafo mixto se puede convertir en dirigido reemplazando cada arista no dirigida por dos dirigidas).

![Grafo no dirigido y grafo dirigido](../assets/ch14_grafo_intro.png)

**Ejemplos de la vida real:** la red de vuelos entre aeropuertos (dirigido: hay vuelos de ida que no tienen vuelta), el mapa de calles de una ciudad (mixto: hay calles de un solo sentido), las amistades de una red social (no dirigido), las dependencias entre materias de una carrera (dirigido y sin ciclos), las páginas web y sus enlaces (dirigido).

### Vocabulario

| Término | Significado |
|---|---|
| **Extremos** (*end vertices*) | Los dos vértices que une una arista. En una arista dirigida: **origen** y **destino** |
| **Adyacentes** | $u$ y $v$ son adyacentes si existe una arista entre ellos |
| **Incidente** | Una arista es incidente a un vértice si el vértice es uno de sus extremos |
| **Aristas salientes / entrantes** | En un digrafo: las aristas cuyo origen / destino es $v$ |
| **Grado** $\deg(v)$ | Número de aristas incidentes a $v$ |
| **Grado de entrada / salida** | $\text{indeg}(v)$ y $\text{outdeg}(v)$: número de aristas entrantes / salientes de $v$ |
| **Aristas paralelas** | Dos aristas con los mismos extremos (en un digrafo, mismo origen y mismo destino) |
| **Lazo** (*self-loop*) | Arista cuyos dos extremos son el mismo vértice |
| **Grafo simple** | Grafo **sin** aristas paralelas ni lazos. **Goodrich trabaja casi siempre con grafos simples**, y su implementación lo exige |
| **Camino** (*path*) | Secuencia alternada de vértices y aristas que empieza y termina en un vértice; cada arista es incidente a su vértice anterior y siguiente |
| **Ciclo** | Camino con al menos una arista que empieza y termina en el mismo vértice |
| **Camino / ciclo simple** | Sin vértices repetidos (salvo el primero y el último en un ciclo) |
| **Alcanzable** | En un digrafo, $v$ es alcanzable desde $u$ si hay un camino dirigido de $u$ a $v$ |
| **Conexo** | Un grafo no dirigido es conexo si hay un camino entre **todo** par de vértices |
| **Fuertemente conexo** | Un digrafo lo es si para todo par $u, v$, $u$ alcanza a $v$ **y** $v$ alcanza a $u$ |
| **Subgrafo** | Grafo cuyos vértices y aristas son subconjuntos de los de $G$ |
| **Subgrafo de expansión** (*spanning subgraph*) | Subgrafo que contiene **todos** los vértices de $G$ |
| **Bosque** / **árbol** | Un bosque es un grafo sin ciclos; un árbol es un bosque conexo |
| **Árbol de expansión** (*spanning tree*) | Subgrafo de expansión que es un árbol |
| **DAG** | *Directed acyclic graph*: digrafo sin ciclos dirigidos |

![Grado, camino, ciclo y grados de entrada/salida](../assets/ch14_terminologia.png)

## 2. Propiedades básicas

Sea $G$ un grafo con $n$ vértices y $m$ aristas.

**Proposición 14.8.** Si $G$ es no dirigido:
$$\sum_{v \in V} \deg(v) = 2m$$
Cada arista $(u, v)$ se cuenta **dos veces**: una en $\deg(u)$ y otra en $\deg(v)$. En la figura de arriba: $2 + 3 + 3 + 2 + 2 = 12 = 2 \cdot 6$.

**Proposición 14.9.** Si $G$ es dirigido:
$$\sum_{v \in V} \text{indeg}(v) = \sum_{v \in V} \text{outdeg}(v) = m$$
Cada arista dirigida aporta exactamente 1 al grado de salida de su origen y 1 al grado de entrada de su destino.

**Proposición 14.10.** Si $G$ es **simple**:
- no dirigido: $m \le \dfrac{n(n-1)}{2}$ (a lo sumo una arista por cada par de vértices),
- dirigido: $m \le n(n-1)$ (a lo sumo una arista en cada sentido por cada par).

Es decir, $m$ es $O(n^2)$. Un grafo con $m$ cercano a $n^2$ se llama **denso**; uno con $m$ mucho menor (por ejemplo $O(n)$) se llama **disperso** (*sparse*). Esta distinción decide qué representación conviene (sección 4).

**Proposición 14.11.** Si $G$ es no dirigido con $n$ vértices:
- si $G$ es conexo, $m \ge n - 1$;
- si $G$ es un árbol, $m = n - 1$;
- si $G$ es un bosque, $m \le n - 1$.

## 3. El ADT `Graph`

Goodrich modela un grafo con **tres** tipos de objetos: vértices, aristas y el grafo. Los vértices y las aristas son **posiciones** livianas que guardan un elemento (como las `Position` de las listas y árboles): el usuario las recibe al insertar y luego las pasa como parámetro a los métodos del grafo.

### `Vertex`

| Método | Descripción |
|---|---|
| `v.element()` | Retorna el elemento guardado en el vértice (por ejemplo, el nombre de un aeropuerto) |

### `Edge`

| Método | Descripción |
|---|---|
| `e.element()` | Retorna el elemento guardado en la arista (por ejemplo, un número de vuelo o un **peso**) |
| `e.endpoints()` | Retorna la tupla `(u, v)` con los extremos. Si es dirigida, `u` es el origen y `v` el destino |
| `e.opposite(v)` | Dado un extremo `v`, retorna el **otro** extremo |

### `Graph`

| Método | Descripción |
|---|---|
| `G.vertex_count()` | Número de vértices $n$ |
| `G.vertices()` | Iteración de todos los vértices |
| `G.edge_count()` | Número de aristas $m$ |
| `G.edges()` | Iteración de todas las aristas |
| `G.get_edge(u, v)` | Retorna la arista de `u` a `v` si existe; si no, `None`. En un grafo no dirigido, `get_edge(u, v)` y `get_edge(v, u)` dan lo mismo |
| `G.degree(v, out=True)` | Grado de salida de `v`; con `out=False`, grado de entrada. En un grafo no dirigido ambos son $\deg(v)$ |
| `G.incident_edges(v, out=True)` | Iteración de las aristas salientes de `v` (con `out=False`, las entrantes). En un grafo no dirigido, todas las incidentes |
| `G.insert_vertex(x=None)` | Crea y retorna un nuevo `Vertex` con elemento `x` |
| `G.insert_edge(u, v, x=None)` | Crea y retorna una nueva `Edge` de `u` a `v` con elemento `x` |
| `G.remove_vertex(v)` | Elimina `v` **y todas sus aristas incidentes** |
| `G.remove_edge(e)` | Elimina la arista `e` |

En la implementación de Goodrich el parámetro se llama `outgoing` en lugar de `out`. Además, `remove_vertex` y `remove_edge` **no** vienen implementados (quedan como ejercicio).

## 4. Estructuras de datos para grafos

Hay cuatro formas clásicas de guardar un grafo. Las cuatro guardan la colección de vértices; la diferencia está en **cómo se organizan las aristas**. Usamos el grafo de la Figura 14.4 de Goodrich: vértices `u, v, w, z` y aristas `e, f, g, h`.

### 4.1 Lista de aristas (*edge list*)

Una lista `V` con los vértices y una lista `E` con las aristas, sin más organización. Cada arista conoce a sus extremos, pero **un vértice no sabe cuáles son sus aristas**.

- `get_edge(u, v)`, `degree(v)` e `incident_edges(v)` deben **recorrer toda la lista** `E`: $O(m)$.
- `remove_vertex(v)` también es $O(m)$: hay que buscar todas las aristas que tocan a `v`.
- Es la más simple, pero lenta para las consultas que hacen los algoritmos de grafos.

### 4.2 Lista de adyacencia (*adjacency list*)

Además de lo anterior, cada vértice $v$ guarda una lista $I(v)$ con **sus** aristas incidentes (en un digrafo, dos listas: salientes y entrantes).

- `degree(v)` es $O(1)$ (la longitud de $I(v)$) e `incident_edges(v)` es $O(\deg(v))$: solo se recorren las aristas que importan.
- `get_edge(u, v)` debe recorrer $I(u)$ (o la más corta entre $I(u)$ e $I(v)$): $O(\min(\deg(u), \deg(v)))$.
- Memoria: $O(n + m)$, porque $\sum_v |I(v)| = 2m$ (Proposición 14.8).

![Lista de aristas y lista de adyacencia](../assets/ch14_rep_listas.png)

### 4.3 Mapa de adyacencia (*adjacency map*)

Igual que la lista de adyacencia, pero $I(v)$ es un **diccionario** (tabla hash, capítulo 10) cuya **clave es el vértice vecino** y cuyo **valor es la arista** que los une.

- Conserva todo lo bueno de la lista de adyacencia: `degree(v)` en $O(1)$, `incident_edges(v)` en $O(\deg(v))$, memoria $O(n + m)$.
- Y además `get_edge(u, v)` es simplemente `I(u)[v]`: **$O(1)$ esperado**.
- **Es la representación que implementa Goodrich**, usando `dict` de Python.

### 4.4 Matriz de adyacencia (*adjacency matrix*)

Se numeran los vértices $0, 1, \dots, n-1$ y se guarda una matriz $A$ de $n \times n$ donde `A[i][j]` es la arista entre los vértices $i$ y $j$ (o `None` si no son adyacentes). En un grafo no dirigido la matriz es simétrica.

- `get_edge(u, v)` es $O(1)$ **en el peor caso**: un acceso a la matriz.
- Pero `degree(v)` e `incident_edges(v)` deben recorrer **toda** una fila: $O(n)$, aunque $v$ tenga pocos vecinos.
- Memoria $O(n^2)$, aunque el grafo tenga muy pocas aristas, e `insert_vertex` es $O(n^2)$ porque hay que agrandar la matriz.
- Conviene solo para grafos **densos**.

![Mapa de adyacencia y matriz de adyacencia](../assets/ch14_rep_mapa_matriz.png)

### 4.5 Comparación (Tabla 14.1 de Goodrich)

Para un grafo con $n$ vértices y $m$ aristas:

| Operación | Lista de aristas | Lista de adyacencia | Mapa de adyacencia | Matriz de adyacencia |
|---|---|---|---|---|
| `vertex_count()`, `edge_count()` | $O(1)$ | $O(1)$ | $O(1)$ | $O(1)$ |
| `vertices()` | $O(n)$ | $O(n)$ | $O(n)$ | $O(n)$ |
| `edges()` | $O(m)$ | $O(m)$ | $O(m)$ | $O(m)$ |
| `get_edge(u, v)` | $O(m)$ | $O(\min(d_u, d_v))$ | $O(1)$ esp. | $O(1)$ |
| `degree(v)` | $O(m)$ | $O(1)$ | $O(1)$ | $O(n)$ |
| `incident_edges(v)` | $O(m)$ | $O(d_v)$ | $O(d_v)$ | $O(n)$ |
| `insert_vertex(x)` | $O(1)$ | $O(1)$ | $O(1)$ | $O(n^2)$ |
| `remove_vertex(v)` | $O(m)$ | $O(d_v)$ | $O(d_v)$ | $O(n^2)$ |
| `insert_edge(u, v, x)` | $O(1)$ | $O(1)$ | $O(1)$ esp. | $O(1)$ |
| `remove_edge(e)` | $O(1)$ | $O(1)$ | $O(1)$ esp. | $O(1)$ |
| **Memoria** | $O(n + m)$ | $O(n + m)$ | $O(n + m)$ | $O(n^2)$ |

($d_v$ = grado de $v$; "esp." = esperado, por usar tablas hash.)

**Conclusión:** el mapa de adyacencia logra el mejor tiempo (o empata) en casi todas las operaciones, con memoria proporcional al tamaño del grafo.

## 5. La implementación de Goodrich: mapa de adyacencia

La clase `Graph` de Goodrich (Códigos 14.1–14.3) guarda **dos diccionarios de diccionarios**:

- `self._outgoing[u]` es un dict `{v: arista}` con las aristas que **salen** de `u`. Así, `self._outgoing[u][v]` es la arista $u \to v$.
- `self._incoming[v]` es un dict `{u: arista}` con las aristas que **llegan** a `v`. La **misma** arista $u \to v$ está también en `self._incoming[v][u]`.

**Truco para grafos no dirigidos:** no se crea un segundo diccionario; `self._incoming` es **el mismo objeto** que `self._outgoing` (un alias). Entonces `insert_edge(u, v)` guarda la arista en `_outgoing[u][v]` y en `_outgoing[v][u]`, que es justo lo que se necesita: la arista aparece en el mapa de **ambos** extremos. Por eso `is_directed()` se responde comparando si los dos diccionarios son el mismo objeto.

![_outgoing e _incoming en un grafo dirigido](../assets/ch14_goodrich_mapas.png)

**Detalles a observar en el código:**

- `Vertex` y `Edge` usan `__slots__` (objetos livianos, como `_Node` en las listas enlazadas) y definen `__hash__` para poder usarse como **claves** de diccionarios y elementos de conjuntos. Los algoritmos del capítulo guardan información por vértice en diccionarios: `discovered[v]`, `d[v]`, etc.
- `edges()` retorna un **conjunto**: en un grafo no dirigido cada arista aparece en dos mapas secundarios, y el `set` evita reportarla dos veces.
- `edge_count()` suma los tamaños de los mapas secundarios y, si el grafo es no dirigido, divide entre 2 (Proposición 14.8).
- `insert_edge` rechaza aristas paralelas: el grafo es **simple**.

En el código original, `opposite` termina con un `raise ValueError` que nunca se ejecuta (está después de un `return`). Aquí lo corregimos para que sí verifique que `v` sea un extremo de la arista.

In [2]:
class Graph:
    """Grafo simple representado con un mapa de adyacencia (Goodrich, Códigos 14.1-14.3)."""

    # ------------------------- clase anidada Vertex -------------------------
    class Vertex:
        """Estructura liviana para un vértice."""
        __slots__ = '_element'

        def __init__(self, x):
            """No llamar directamente: usar insert_vertex(x) del grafo."""
            self._element = x

        def element(self):
            """Retorna el elemento guardado en el vértice."""
            return self._element

        def __hash__(self):              # permite usar el vértice como clave de dict / elemento de set
            return hash(id(self))

        def __repr__(self):
            return str(self._element)

    # ------------------------- clase anidada Edge -------------------------
    class Edge:
        """Estructura liviana para una arista."""
        __slots__ = '_origin', '_destination', '_element'

        def __init__(self, u, v, x):
            """No llamar directamente: usar insert_edge(u, v, x) del grafo."""
            self._origin = u
            self._destination = v
            self._element = x

        def endpoints(self):
            """Retorna la tupla (u, v) con los extremos de la arista."""
            return (self._origin, self._destination)

        def opposite(self, v):
            """Retorna el vértice del otro extremo de la arista."""
            if not isinstance(v, Graph.Vertex):
                raise TypeError('v debe ser un Vertex')
            if v is self._origin:
                return self._destination
            if v is self._destination:
                return self._origin
            raise ValueError('v no es extremo de esta arista')

        def element(self):
            """Retorna el elemento guardado en la arista."""
            return self._element

        def __hash__(self):              # permite usar la arista como clave de dict / elemento de set
            return hash((self._origin, self._destination))

        def __repr__(self):
            return f'({self._origin},{self._destination},{self._element})'

    # ------------------------- métodos de Graph -------------------------
    def __init__(self, directed=False):
        """Crea un grafo vacío (no dirigido por defecto)."""
        self._outgoing = {}
        # solo un digrafo necesita un segundo mapa; en uno no dirigido _incoming es un alias
        self._incoming = {} if directed else self._outgoing

    def _validate_vertex(self, v):
        """Verifica que v sea un Vertex de este grafo."""
        if not isinstance(v, self.Vertex):
            raise TypeError('Se esperaba un Vertex')
        if v not in self._outgoing:
            raise ValueError('El vértice no pertenece a este grafo')

    def is_directed(self):
        """True si el grafo es dirigido (depende de cómo se creó, no de su contenido)."""
        return self._incoming is not self._outgoing     # dirigido si los mapas son distintos

    def vertex_count(self):
        """Número de vértices."""
        return len(self._outgoing)

    def vertices(self):
        """Iteración de todos los vértices."""
        return self._outgoing.keys()

    def edge_count(self):
        """Número de aristas."""
        total = sum(len(self._outgoing[v]) for v in self._outgoing)
        # en un grafo no dirigido cada arista se contó dos veces
        return total if self.is_directed() else total // 2

    def edges(self):
        """Conjunto de todas las aristas."""
        result = set()                   # el set evita repetir aristas en grafos no dirigidos
        for secondary_map in self._outgoing.values():
            result.update(secondary_map.values())
        return result

    def get_edge(self, u, v):
        """Retorna la arista de u a v, o None si no son adyacentes."""
        self._validate_vertex(u)
        self._validate_vertex(v)
        return self._outgoing[u].get(v)  # None si v no es vecino de u

    def degree(self, v, outgoing=True):
        """Número de aristas salientes de v (entrantes si outgoing=False)."""
        self._validate_vertex(v)
        adj = self._outgoing if outgoing else self._incoming
        return len(adj[v])

    def incident_edges(self, v, outgoing=True):
        """Iteración de las aristas salientes de v (entrantes si outgoing=False)."""
        self._validate_vertex(v)
        adj = self._outgoing if outgoing else self._incoming
        for edge in adj[v].values():
            yield edge

    def insert_vertex(self, x=None):
        """Inserta y retorna un nuevo Vertex con elemento x."""
        v = self.Vertex(x)
        self._outgoing[v] = {}
        if self.is_directed():
            self._incoming[v] = {}       # un digrafo necesita también su mapa de entrantes
        return v

    def insert_edge(self, u, v, x=None):
        """Inserta y retorna una nueva Edge de u a v con elemento x.

        Lanza ValueError si u y v ya son adyacentes.
        """
        if self.get_edge(u, v) is not None:     # incluye la validación de u y v
            raise ValueError('u y v ya son adyacentes')
        e = self.Edge(u, v, x)
        self._outgoing[u][v] = e
        self._incoming[v][u] = e
        return e

## 6. Ejemplos de uso

### 6.1 Grafo no dirigido, paso a paso

Construimos el grafo no dirigido de la sección 1: vértices `A, B, C, D, E` y aristas `A–B, A–C, B–C, B–D, C–E, D–E`. A cada arista le ponemos como elemento un texto con su nombre.

Observa que `insert_vertex` **retorna** el vértice: hay que guardarlo, porque los demás métodos reciben vértices, no etiquetas.

In [3]:
G = Graph()                                  # no dirigido

A = G.insert_vertex('A')
B = G.insert_vertex('B')
C = G.insert_vertex('C')
D = G.insert_vertex('D')
E = G.insert_vertex('E')

for u, v in [(A, B), (A, C), (B, C), (B, D), (C, E), (D, E)]:
    G.insert_edge(u, v, f'{u}{v}')

print(G.is_directed())                       # False
print(G.vertex_count(), G.edge_count())      # 5 6
print(list(G.vertices()))                    # [A, B, C, D, E]
print(sorted(G.edges(), key=str))            # [(A,B,AB), (A,C,AC), (B,C,BC), (B,D,BD), (C,E,CE), (D,E,DE)]

False
5 6
[A, B, C, D, E]
[(A,B,AB), (A,C,AC), (B,C,BC), (B,D,BD), (C,E,CE), (D,E,DE)]


In [4]:
# --- grado y aristas incidentes ---
for v in G.vertices():
    print(v, 'grado', G.degree(v), '->', list(G.incident_edges(v)))
# A grado 2 -> [(A,B,AB), (A,C,AC)]
# B grado 3 -> [(A,B,AB), (B,C,BC), (B,D,BD)]
# C grado 3 -> [(A,C,AC), (B,C,BC), (C,E,CE)]
# D grado 2 -> [(B,D,BD), (D,E,DE)]
# E grado 2 -> [(C,E,CE), (D,E,DE)]

# Proposición 14.8: la suma de grados es 2m
print(sum(G.degree(v) for v in G.vertices()) == 2 * G.edge_count())   # True

A grado 2 -> [(A,B,AB), (A,C,AC)]
B grado 3 -> [(A,B,AB), (B,C,BC), (B,D,BD)]
C grado 3 -> [(A,C,AC), (B,C,BC), (C,E,CE)]
D grado 2 -> [(B,D,BD), (D,E,DE)]
E grado 2 -> [(C,E,CE), (D,E,DE)]
True


In [5]:
# --- vecinos: opposite convierte cada arista incidente en el vértice del otro lado ---
vecinos_B = [e.opposite(B) for e in G.incident_edges(B)]
print(vecinos_B)                             # [A, C, D]

# --- get_edge: en un grafo no dirigido el orden de los extremos no importa ---
print(G.get_edge(B, D))                      # (B,D,BD)
print(G.get_edge(D, B))                      # (B,D,BD)   <- la MISMA arista
print(G.get_edge(D, B) is G.get_edge(B, D))  # True
print(G.get_edge(A, E))                      # None: A y E no son adyacentes

e = G.get_edge(C, E)
print(e.endpoints(), e.element())            # (C, E) CE

[A, C, D]
(B,D,BD)
(B,D,BD)
True
None
(C, E) CE


In [6]:
# --- el grafo es simple: no admite aristas paralelas ---
try:
    G.insert_edge(E, D, 'otra')              # D-E ya existe (no importa el orden)
except ValueError as err:
    print('ValueError:', err)                # ValueError: u y v ya son adyacentes

# --- los vértices de OTRO grafo no sirven ---
H = Graph()
X = H.insert_vertex('X')
try:
    G.degree(X)
except ValueError as err:
    print('ValueError:', err)                # ValueError: El vértice no pertenece a este grafo

# --- opposite con un vértice que no es extremo ---
try:
    G.get_edge(A, B).opposite(E)
except ValueError as err:
    print('ValueError:', err)                # ValueError: v no es extremo de esta arista

ValueError: u y v ya son adyacentes
ValueError: El vértice no pertenece a este grafo
ValueError: v no es extremo de esta arista


In [7]:
# --- por dentro: en un grafo no dirigido _incoming ES _outgoing ---
print(G._incoming is G._outgoing)            # True
for v, mapa in G._outgoing.items():
    print(v, '->', mapa)
# A -> {B: (A,B,AB), C: (A,C,AC)}
# B -> {A: (A,B,AB), C: (B,C,BC), D: (B,D,BD)}
# C -> {A: (A,C,AC), B: (B,C,BC), E: (C,E,CE)}
# D -> {B: (B,D,BD), E: (D,E,DE)}
# E -> {C: (C,E,CE), D: (D,E,DE)}

True
A -> {B: (A,B,AB), C: (A,C,AC)}
B -> {A: (A,B,AB), C: (B,C,BC), D: (B,D,BD)}
C -> {A: (A,C,AC), B: (B,C,BC), E: (C,E,CE)}
D -> {B: (B,D,BD), E: (D,E,DE)}
E -> {C: (C,E,CE), D: (D,E,DE)}


Cada arista aparece **dos veces** en el mapa (una por cada extremo), pero es **el mismo objeto**: `(A,B,AB)` está en el mapa de `A` con clave `B` y en el mapa de `B` con clave `A`. Por eso `edge_count()` divide entre 2 y `edges()` usa un `set`.

### 6.2 Grafo dirigido y ponderado

Ahora el digrafo de la figura de la sección 5: `A → B` (peso 5), `A → C` (peso 2), `C → B` (peso 1). El **peso** es simplemente el elemento de la arista; así es como los algoritmos de caminos más cortos y árboles de expansión mínima de Goodrich leen los pesos: `e.element()`.

In [8]:
DG = Graph(directed=True)
a = DG.insert_vertex('A')
b = DG.insert_vertex('B')
c = DG.insert_vertex('C')
DG.insert_edge(a, b, 5)
DG.insert_edge(a, c, 2)
DG.insert_edge(c, b, 1)

print(DG.is_directed())                      # True
print(DG.vertex_count(), DG.edge_count())    # 3 3   <- ya no se divide entre 2

for v in DG.vertices():
    print(v, 'outdeg =', DG.degree(v), ' indeg =', DG.degree(v, False))
# A outdeg = 2  indeg = 0
# B outdeg = 0  indeg = 2
# C outdeg = 1  indeg = 1

True
3 3
A outdeg = 2  indeg = 0
B outdeg = 0  indeg = 2
C outdeg = 1  indeg = 1


In [9]:
print(list(DG.incident_edges(b)))            # []                      <- de B no sale nada
print(list(DG.incident_edges(b, False)))     # [(A,B,5), (C,B,1)]      <- a B llegan dos

# en un digrafo el orden SÍ importa
print(DG.get_edge(a, b))                     # (A,B,5)
print(DG.get_edge(b, a))                     # None

# peso total de las aristas
print(sum(e.element() for e in DG.edges()))  # 8

# los dos mapas por dentro (compara con la figura de la sección 5)
print(DG._outgoing)                          # {A: {B: (A,B,5), C: (A,C,2)}, B: {}, C: {B: (C,B,1)}}
print(DG._incoming)                          # {A: {}, B: {A: (A,B,5), C: (C,B,1)}, C: {A: (A,C,2)}}

[]
[(A,B,5), (C,B,1)]
(A,B,5)
None
8
{A: {B: (A,B,5), C: (A,C,2)}, B: {}, C: {B: (C,B,1)}}
{A: {}, B: {A: (A,B,5), C: (C,B,1)}, C: {A: (A,C,2)}}


### 6.3 Construir un grafo a partir de una lista de aristas

Escribir `insert_vertex` e `insert_edge` uno por uno es tedioso. Goodrich trae una función auxiliar (`graph_examples.py`) que recibe una secuencia de tuplas `(origen, destino)` o `(origen, destino, elemento)` y arma el grafo: primero crea un vértice por cada etiqueta distinta y luego inserta las aristas.

**Cambio respecto al original:** Goodrich junta las etiquetas en un `set`, y el orden de un `set` de cadenas cambia en cada sesión de Python (igual que `hash()` de cadenas, capítulo 10). Para que todos veamos los vértices en el mismo orden, usamos un `dict` (que conserva el orden de inserción). También retornamos el diccionario `etiqueta -> Vertex`, para poder pedir un vértice por su nombre.

In [10]:
def graph_from_edgelist(E, directed=False):
    """Construye un Graph a partir de tuplas (origen, destino) o (origen, destino, elemento).

    Retorna (g, verts) donde verts es un dict etiqueta -> Vertex.
    """
    g = Graph(directed)
    etiquetas = {}                           # dict como "conjunto ordenado"
    for e in E:
        etiquetas[e[0]] = None
        etiquetas[e[1]] = None

    verts = {}                               # etiqueta -> Vertex
    for x in etiquetas:
        verts[x] = g.insert_vertex(x)

    for e in E:
        elemento = e[2] if len(e) > 2 else None
        g.insert_edge(verts[e[0]], verts[e[1]], elemento)
    return g, verts


# el grafo ponderado que usaremos en los próximos notebooks (DFS, BFS, Dijkstra, MST)
PONDERADO = [('A', 'B', 4), ('A', 'C', 2), ('B', 'C', 1), ('B', 'D', 5), ('C', 'D', 8),
             ('C', 'E', 10), ('D', 'E', 2), ('D', 'F', 6), ('E', 'F', 3)]
GP, VP = graph_from_edgelist(PONDERADO)
print(GP.vertex_count(), GP.edge_count())    # 6 9
print(list(GP.incident_edges(VP['D'])))      # [(B,D,5), (C,D,8), (D,E,2), (D,F,6)]
print(GP.get_edge(VP['F'], VP['E']))         # (E,F,3)

6 9
[(B,D,5), (C,D,8), (D,E,2), (D,F,6)]
(E,F,3)


Este es el grafo ponderado de las figuras de DFS/BFS, Dijkstra y árbol de expansión mínima que veremos más adelante:

![Grafo ponderado de ejemplo](../assets/ch14_mst.png)

### 6.4 Ejemplo del libro: vuelos entre aeropuertos (Figura 14.3)

Un digrafo donde los vértices son aeropuertos y cada arista dirigida es un vuelo.

![Grafo de vuelos de la Figura 14.3](../assets/ch14_aeropuertos.png)

In [11]:
VUELOS = (
    ('BOS', 'SFO'), ('BOS', 'JFK'), ('BOS', 'MIA'), ('JFK', 'BOS'),
    ('JFK', 'DFW'), ('JFK', 'MIA'), ('JFK', 'SFO'), ('ORD', 'DFW'),
    ('ORD', 'MIA'), ('LAX', 'ORD'), ('DFW', 'SFO'), ('DFW', 'ORD'),
    ('DFW', 'LAX'), ('MIA', 'DFW'), ('MIA', 'LAX'),
)
vuelos, aero = graph_from_edgelist(VUELOS, directed=True)

print(vuelos.vertex_count(), vuelos.edge_count())        # 7 15

# ¿A dónde se puede volar directo desde JFK?
print([e.opposite(aero['JFK']) for e in vuelos.incident_edges(aero['JFK'])])
# [BOS, DFW, MIA, SFO]

# ¿Desde dónde llegan vuelos directos a MIA?
print([e.opposite(aero['MIA']) for e in vuelos.incident_edges(aero['MIA'], False)])
# [BOS, JFK, ORD]

# ¿Hay vuelo directo ORD -> SFO? ¿Y DFW -> SFO?
print(vuelos.get_edge(aero['ORD'], aero['SFO']))         # None
print(vuelos.get_edge(aero['DFW'], aero['SFO']))         # (DFW,SFO,None)

7 15
[BOS, DFW, MIA, SFO]
[BOS, JFK, ORD]
None
(DFW,SFO,None)


In [12]:
# Proposición 14.9: suma de indeg = suma de outdeg = m
print(f'{"aeropuerto":>10} {"outdeg":>7} {"indeg":>6}')
for v in vuelos.vertices():
    print(f'{v.element():>10} {vuelos.degree(v):>7} {vuelos.degree(v, False):>6}')
print(sum(vuelos.degree(v) for v in vuelos.vertices()),
      sum(vuelos.degree(v, False) for v in vuelos.vertices()))   # 15 15
#  aeropuerto  outdeg  indeg
#         BOS       3      1
#         SFO       0      3
#         JFK       4      1
#         MIA       2      3
#         DFW       3      3
#         ORD       2      2
#         LAX       1      2

# SFO no tiene vuelos de salida: una vez allí no se puede ir a ninguna parte
print([v for v in vuelos.vertices() if vuelos.degree(v) == 0])   # [SFO]

aeropuerto  outdeg  indeg
       BOS       3      1
       SFO       0      3
       JFK       4      1
       MIA       2      3
       DFW       3      3
       ORD       2      2
       LAX       1      2
15 15
[SFO]


### 6.5 De mapa de adyacencia a matriz de adyacencia

Para relacionar las representaciones de la sección 4, construimos la matriz de adyacencia de un `Graph`: numeramos los vértices $0, \dots, n-1$ y llenamos `A[i][j]` con `1` si existe la arista $i \to j$. Cada consulta `get_edge` es $O(1)$ esperado, así que construir la matriz cuesta $O(n^2)$.

In [13]:
def matriz_adyacencia(g):
    """Retorna (orden, A): la lista de vértices y la matriz 0/1 de adyacencia de g."""
    orden = list(g.vertices())
    n = len(orden)
    A = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j and g.get_edge(orden[i], orden[j]) is not None:
                A[i][j] = 1
    return orden, A


def imprimir_matriz(orden, A):
    print('     ' + ' '.join(f'{str(v):>4}' for v in orden))
    for v, fila in zip(orden, A):
        print(f'{str(v):>4} ' + ' '.join(f'{x:>4}' for x in fila))


imprimir_matriz(*matriz_adyacencia(G))       # no dirigido: la matriz es simétrica
#         A    B    C    D    E
#    A    0    1    1    0    0
#    B    1    0    1    1    0
#    C    1    1    0    0    1
#    D    0    1    0    0    1
#    E    0    0    1    1    0

        A    B    C    D    E
   A    0    1    1    0    0
   B    1    0    1    1    0
   C    1    1    0    0    1
   D    0    1    0    0    1
   E    0    0    1    1    0


In [14]:
imprimir_matriz(*matriz_adyacencia(vuelos))  # dirigido: la fila es el origen, la columna el destino
#       BOS  SFO  JFK  MIA  DFW  ORD  LAX
#  BOS    0    1    1    1    0    0    0
#  SFO    0    0    0    0    0    0    0
#  JFK    1    1    0    1    1    0    0
#  MIA    0    0    0    0    1    0    1
#  DFW    0    1    0    0    0    1    1
#  ORD    0    0    0    1    1    0    0
#  LAX    0    0    0    0    0    1    0

# la suma de una fila es el outdeg; la de una columna, el indeg
orden, M = matriz_adyacencia(vuelos)
print([sum(fila) for fila in M])                                 # [3, 0, 4, 2, 3, 2, 1]
print([sum(M[i][j] for i in range(len(M))) for j in range(len(M))])   # [1, 3, 1, 3, 3, 2, 2]

      BOS  SFO  JFK  MIA  DFW  ORD  LAX
 BOS    0    1    1    1    0    0    0
 SFO    0    0    0    0    0    0    0
 JFK    1    1    0    1    1    0    0
 MIA    0    0    0    0    1    0    1
 DFW    0    1    0    0    0    1    1
 ORD    0    0    0    1    1    0    0
 LAX    0    0    0    0    0    1    0
[3, 0, 4, 2, 3, 2, 1]
[1, 3, 1, 3, 3, 2, 2]


## 7. Resumen

- Un grafo es un conjunto de **vértices** y una colección de **aristas**; las aristas pueden ser **dirigidas** o **no dirigidas**. Goodrich trabaja con grafos **simples** (sin aristas paralelas ni lazos).
- $\sum \deg(v) = 2m$ en un grafo no dirigido; $\sum \text{indeg}(v) = \sum \text{outdeg}(v) = m$ en uno dirigido; y $m = O(n^2)$.
- El ADT tiene tres tipos de objetos: `Vertex` (`element`), `Edge` (`element`, `endpoints`, `opposite`) y `Graph` (`vertex_count`, `vertices`, `edge_count`, `edges`, `get_edge`, `degree`, `incident_edges`, `insert_vertex`, `insert_edge`, `remove_vertex`, `remove_edge`).
- Hay cuatro representaciones: **lista de aristas**, **lista de adyacencia**, **mapa de adyacencia** y **matriz de adyacencia**. El mapa de adyacencia da `get_edge` en $O(1)$ esperado, `degree` en $O(1)$, `incident_edges(v)` en $O(\deg(v))$ y memoria $O(n + m)$.
- La clase `Graph` de Goodrich guarda `_outgoing[u][v]` y `_incoming[v][u]` apuntando a la **misma** arista; en un grafo no dirigido `_incoming` es un alias de `_outgoing`.

**Patrón que se repite en todos los algoritmos del capítulo:**

```python
for e in g.incident_edges(u):     # cada arista que sale de u
    v = e.opposite(u)             # el vecino al otro lado
    peso = e.element()            # (si el grafo es ponderado)
    ...
```

**Para pensar:**

1. Implementa `remove_edge(e)`: ¿de qué diccionarios hay que borrar la arista? ¿Cambia algo si el grafo es no dirigido?
2. Implementa `remove_vertex(v)` en $O(\deg(v))$. Pista: antes de borrar `v`, recorre sus aristas salientes **y** entrantes y bórralas del mapa del vecino.
3. ¿Qué pasa con `edge_count()` si se permitiera un lazo $(v, v)$ en un grafo no dirigido?